# MotorGuard — Egyptian Arabic voice clip renderer

Renders all 102 fault-assistant sentences through **Coqui XTTS-v2** (zero-shot voice cloning,
official Arabic support).

This version replaces the app's original reference voice — pick a **female** voice one of two ways:

- **Option A — upload your own reference clip** (recommended if you have a specific voice in mind):
  10–30s of clean speech, any language, one speaker, no background noise/music.
- **Option B — preview XTTS's built-in preset voices** and pick one by ear (run the preview cell,
  download the zip, listen, then set `SPEAKER_NAME` below to whichever one you like).

Only run ONE of Option A or Option B — whichever cell you run last decides which voice gets used below.

**Runtime > Change runtime type > GPU (T4 is fine)** before running.


In [ ]:
!pip install -q TTS


In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"  # accept the model license non-interactively

from TTS.api import TTS
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

SPEAKER_WAV = None  # Option A sets this
SPEAKER_NAME = None  # Option B sets this


## Option A — upload your own reference clip
Skip this cell if you're using Option B instead.


In [ ]:
from google.colab import files
import IPython.display as ipd

uploaded = files.upload()  # pick your reference audio file
SPEAKER_WAV = next(iter(uploaded))
SPEAKER_NAME = None
print("using uploaded reference:", SPEAKER_WAV)
ipd.Audio(SPEAKER_WAV)


## Option B — preview XTTS's built-in preset voices
Skip this if you already picked Option A. This synthesizes a short Arabic test line with every
built-in preset voice, zips the previews, and downloads them so you can listen and choose one by name.


In [ ]:
import pathlib, shutil
from google.colab import files as colab_files

print(len(tts.speakers), "preset voices:", tts.speakers)

pathlib.Path("previews").mkdir(exist_ok=True)
TEST_LINE = "المحرك بتاعك سخن أكتر من اللازم"

for name in tts.speakers:
    safe = name.replace(" ", "_")
    tts.tts_to_file(text=TEST_LINE, speaker=name, language="ar", file_path=f"previews/{safe}.wav")
    print("rendered", name)

shutil.make_archive("voice_previews", "zip", "previews")
colab_files.download("voice_previews.zip")


In [ ]:
# After listening to the previews, set the name you liked (must match tts.speakers exactly) and re-run this cell:
SPEAKER_NAME = "REPLACE_WITH_CHOSEN_NAME"
SPEAKER_WAV = None
assert SPEAKER_NAME in tts.speakers, "that name isn't in tts.speakers -- copy it exactly from the printed list above"


## Sanity check

Renders two sentences with whichever voice you picked above (Option A's `SPEAKER_WAV` or Option B's
`SPEAKER_NAME`) so you can confirm it sounds right before rendering all 102.


In [ ]:
import pathlib
import IPython.display as ipd

assert SPEAKER_WAV or SPEAKER_NAME, "run Option A or Option B above first"

SENTENCES = {
  "p0217_name": "حرارة المحرك زايدة عن الحد",
  "p0217_expl": "المحرك بتاعك سخن أكتر من اللازم.",
  "p0217_expl_2": "غالبًا السبب مياه التبريد ناقصة، أو في تسريب في نظام التبريد، أو طرمبة المية أو الترموستات بايظة.",
  "p0217_expl_3": "لو استمريت تسوق والعربية سخنة كده، ممكن تتلف المحرك بشكل دائم.",
  "p0217_action": "قلل من سرعتك ودوّر على مكان آمن توقف فيه بسرعة.",
  "p0217_action_2": "لو مؤشر الحرارة وصل للمنطقة الحمرا، قف على جنب واطفي المحرك فورًا.",
  "p0524_name": "ضغط زيت المحرك واطي جدًا",
  "p0524_expl": "ضغط الزيت نزل تحت المعدل الآمن.",
  "p0524_expl_2": "من غير ضغط زيت كويس، المحرك ممكن يقف تمامًا خلال دقايق.",
  "p0524_expl_3": "ده من أخطر التحذيرات اللي ممكن العربية تديهالك.",
  "p0524_action": "قف على جنب بأمان واطفي المحرك أول ما تقدر.",
  "p0524_action_2": "متكملش السواقة خالص.",
  "c1201_name": "عطل في نظام التحكم بالثبات",
  "c1201_expl": "نظام التحكم في الثبات والجر حصله عطل ودلوقتي مش شغال.",
  "c1201_expl_2": "العربية لسه بتقود وبتفرمل عادي، بس مش هتساعدك تلقائيًا لو عجلة زحلقت.",
  "c1201_action": "سوق بهدوء، خصوصًا في الشتا أو الطريق المبلول، وودّي العربية تتفحص في أقرب يوم أو يومين.",
  "c0035_name": "حساس سرعة عجلة الشمال الأمامية بايظ",
  "c0035_expl": "حساس سرعة العجلة بايظ.",
  "c0035_expl_2": "ده ممكن يعطّل نظام الـ ABS والتحكم في الثبات لأن العربية مبقتش تقدر تقيس سرعة دوران العجلة دي.",
  "c0035_expl_3": "الفرامل العادية لسه شغالة كويس.",
  "c0035_action": "تجنب الفرملة القوية على الأرض الزلقة واحجز موعد صيانة قريب.",
  "p0300_name": "تفويت اشتعال في المحرك",
  "p0300_expl": "سلندر واحد أو أكتر بيفوّت الاشتعال.",
  "p0300_expl_2": "ممكن تحس إن المحرك بيهتز أو بيفقد قوته.",
  "p0300_expl_3": "الاستمرار في التفويت لفترة طويلة ممكن يسخّن ويتلف الكاتلينيك.",
  "p0300_action": "خفف من الدوس على البنزين وابعد عن السرعات العالية.",
  "p0300_action_2": "وديها تتشاف النهاردة، خصوصًا لو اللمبة بتلمع مش ثابتة.",
  "p0420_name": "كفاءة الكاتلينيك منخفضة",
  "p0420_expl": "الكاتلينيك مش بينضف العادم زي ما المفروض.",
  "p0420_expl_2": "العربية آمنة إنك تسوقها، بس مش هتعدي فحص الشكمان وممكن استهلاك البنزين يزيد.",
  "p0420_action": "مفيش حاجة مستعجلة.",
  "p0420_action_2": "وديها تتفحص في الصيانة الجاية.",
  "p0562_name": "جهد النظام الكهربائي منخفض",
  "p0562_expl": "جهد النظام الكهربائي واطي، وده غالبًا معناه إن الدينامو أو البطارية بايظة.",
  "p0562_expl_2": "لو الدينامو بايظ، العربية ممكن تفقد الكهربا وتقف لما البطارية تفضى.",
  "p0562_action": "قفل الحاجات الكهربائية اللي مش ضرورية (التكييف، تدفئة الكراسي) وروح لأقرب محطة خدمة.",
  "p0562_action_2": "لو النور بيخفت أو لمبة البطارية شغالة، قف بسرعة.",
  "p0128_name": "حرارة المياه أقل من درجة التشغيل",
  "p0128_expl": "المحرك بياخد وقت طويل عشان يسخن، وغالبًا السبب إن الترموستات عالقة مفتوحة.",
  "p0128_expl_2": "مش خطر، بس بيأثر على استهلاك البنزين والتدفئة جوه العربية.",
  "p0128_action_2": "اذكرها في الصيانة الجاية.",
  "u0100_name": "فقدان الاتصال مع كمبيوتر المحرك",
  "u0100_expl": "وحدة تحكم توقفت عن الاتصال بكمبيوتر المحرك عبر الشبكة الداخلية.",
  "u0100_expl_2": "تصرف العربية ممكن يبقى غير متوقع، واللمبات ممكن تدي إشارات غلط.",
  "u0100_action": "تعامل مع أي تحذير تاني بحذر ووديها تتفحص النهاردة.",
  "b1000_name": "عطل في نظام الوسائد الهوائية",
  "b1000_expl": "في عطل في نظام الوسائد الهوائية وممكن متنفتحش لو حصل حادث.",
  "b1000_expl_2": "العربية بتمشي عادي، بس نظام أمان مهم مش شغال صح.",
  "b1000_action": "سوق بحرص زيادة ووديها نظام الحماية يتفحص أول ما تقدر.",
  "pred_brake_wear_name": "تيل الفرامل قرب يخلص",
  "pred_brake_wear_expl": "نظام الصيانة التنبؤي بيتوقع إن تيل الفرامل قرب على حد الاستهلاك بناءً على سواقتك الأخيرة.",
  "pred_brake_wear_expl_2": "الفرملة تمام دلوقتي، بس التيل هيحتاج تغيير قريب.",
  "pred_brake_wear_action": "احجز كشف فرامل خلال الأسبوعين الجايين.",
  "pred_battery_health_name": "حالة البطارية بتضعف",
  "pred_battery_health_expl": "البطارية بادية عليها علامات كبر في السن وممكن تواجه صعوبة في تشغيل العربية في الجو البارد.",
  "pred_battery_health_expl_2": "لسه مبطلتش شغل خالص.",
  "pred_battery_health_action": "فكر تغيّر البطارية قبل الشتا أو قبل أي رحلة طويلة.",
  "pred_coolant_trend_name": "حرارة المياه بتميل للارتفاع",
  "pred_coolant_trend_expl": "في الرحلات الأخيرة المحرك بيسخن شوية أكتر من المعتاد.",
  "pred_coolant_trend_expl_2": "مفيش حاجة غلط دلوقتي، بس ده غالبًا علامة مبكرة على مشكلة في نظام التبريد.",
  "pred_coolant_trend_action": "ودّي نظام التبريد يتفحص قريب، قبل ما تبقى مشكلة فعلية.",
  "pred_tire_pressure_name": "تسريب بطيء في ضغط الكاوتش",
  "pred_tire_pressure_expl": "في كاوتش واحد بيفقد ضغطه ببطء مع الوقت، وده ممكن يكون سبب ثقب صغير أو صمام بايظ.",
  "pred_tire_pressure_expl_2": "مش نايم خالص، بس الضغط بينزل تدريجيًا.",
  "pred_tire_pressure_action": "افحص وزوّد هوا الكاوتش قريب، وودّيه يتفحص لو فيه ثقب.",
  "rule_coolant_stopnow": "المحرك سخن لدرجة خطيرة جدًا.",
  "rule_coolant_stopnow_2": "قف على جنب بأمان واطفي المحرك فورًا عشان تتجنب تلف دائم.",
  "rule_coolant_trend_soon": "حرارة المحرك بترتفع أسرع من المعتاد.",
  "rule_coolant_trend_soon_2": "ودّي نظام التبريد يتفحص خلال يوم أو يومين.",
  "rule_voltage_urgent": "البطارية بتفضى والعربية ممكن تقف فجأة.",
  "rule_voltage_urgent_2": "روح لأقرب نقطة خدمة دلوقتي وتجنب إنك تطفي المحرك.",
  "rule_misfire_urgent": "التفويت شديد لدرجة إنه ممكن يتلف نظام العادم.",
  "rule_misfire_urgent_2": "قلل السرعة ووديها تتفحص النهاردة.",
  "unrecognised_name": "كود عطل غير معروف",
  "unrecognised_expl": "معنديش تفاصيل عن الكود ده بالتحديد، فمقدرش أشرحهولك بالكامل.",
  "unrecognised_action_urgent": "عشان تبقى مطمن، اتعامل مع الكود ده على إنه خطير ووديها تتشاف أول ما تقدر.",
  "unrecognised_action_routine": "وديها مركز صيانة أول ما يبقى مناسب ليك.",
  "repeat_nothing_said": "لسه مقلتش حاجة.",
  "cancel_ok": "تمام.",
  "unknown_fallback": "معلش، مسمعتش كويس.",
  "unknown_fallback_2": "تقدر تسألني أشرحلك لمبة تحذير، أقولك لو خطيرة، أو أدورلك على أقرب ورشة.",
  "no_active_warnings": "خبر كويس، مفيش أي تحذيرات نشطة دلوقتي.",
  "nothing_active_severity": "مفيش حاجة نشطة تقلقك دلوقتي.",
  "no_location_provider": "مقدرش أدور على ورش قريبة في النسخة دي لسه، بس بناءً على التحذير لازم تودّيها تتشاف قريب.",
  "no_station_nearby": "معرفتش ألاقي محطة خدمة قريبة دلوقتي.",
  "here_is_nearby": "دي أقرب حاجة ليك.",
  "no_faults_everything_fine": "مفيش أي أعطال دلوقتي.",
  "no_faults_everything_fine_2": "كل حاجة تمام.",
  "list_faults_ask_more": "اسألني عن أي واحدة فيهم عشان أشرحلك أكتر.",
  "help_text": "أنا مساعد الصيانة بتاعك.",
  "help_text_2": "تقدر تسألني حاجات زي: إيه اللمبة دي، هل الموضوع خطير، أقدر أكمل سواقة ولا لأ، أو فين أقرب ورشة.",
  "help_text_3": "وهقولك بنفسي لو في حاجة مستعجلة حصلت.",
  "severity_lead_stopnow": "الموضوع مستعجل.",
  "severity_lead_urgent": "الموضوع خطير.",
  "severity_lead_soon": "يستحق إنك تتصرف فيه قريب.",
  "severity_lead_advisory": "الموضوع بسيط.",
  "predicted_heads_up": "ده تنبيه استباقي مش عطل نشط دلوقتي.",
  "assess_stopnow": "لأ — لازم توقف أول ما يبقى آمن.",
  "assess_urgent": "تقدر تسوق بحرص دلوقتي، بس متأجلش الموضوع.",
  "assess_soon": "أيوه، تقدر تكمل سواقة، بس وديها تتشاف قريب.",
  "assess_fine": "أيوه، تمام إنك تكمل سواقة.",
  "assess_escalated": "رفعت درجة الخطورة بسبب قراءات الحساسات الحالية."
}

pathlib.Path("clips").mkdir(exist_ok=True)

def synth_kwargs(text, file_path):
    kw = dict(text=text, language="ar", file_path=file_path)
    if SPEAKER_WAV:
        kw["speaker_wav"] = SPEAKER_WAV
    else:
        kw["speaker"] = SPEAKER_NAME
    return kw

sample_ids = list(SENTENCES)[:2]
for sid in sample_ids:
    tts.tts_to_file(**synth_kwargs(SENTENCES[sid], f"clips/{sid}.wav"))
    print(sid, "->", SENTENCES[sid])

ipd.Audio(f"clips/{sample_ids[0]}.wav")


## Full batch (all 102 sentences)

Only run this after the sanity check sounds right. XTTS on a T4 is roughly a few seconds per
sentence, so this should take well under 20 minutes. Re-running is safe — it just overwrites.

Every clip is resampled to 24kHz mono PCM16 to match `ArabicClipTts.SAMPLE_RATE` and the format the
app expects.


In [ ]:
import soundfile as sf
import librosa
import numpy as np

TARGET_SR = 24000

def render(sid, text):
    raw_path = f"/tmp/_raw_{sid}.wav"
    tts.tts_to_file(**synth_kwargs(text, raw_path))
    audio, sr = sf.read(raw_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != TARGET_SR:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
    pcm16 = np.clip(audio, -1.0, 1.0)
    sf.write(f"clips/{sid}.wav", pcm16, TARGET_SR, subtype="PCM_16")

failed = []
for i, (sid, text) in enumerate(SENTENCES.items(), 1):
    try:
        render(sid, text)
        print(f"[{i}/{len(SENTENCES)}] {sid} ok")
    except Exception as e:
        print(f"[{i}/{len(SENTENCES)}] {sid} FAILED: {e}")
        failed.append(sid)

print("done. failed:", failed)


In [ ]:
import shutil
shutil.make_archive("voice_ar_clips", "zip", "clips")
print("zipped", len(os.listdir("clips")), "files")


In [ ]:
from google.colab import files
files.download("voice_ar_clips.zip")


## After downloading

Unzip `voice_ar_clips.zip` and **replace everything** in `app/src/main/assets/voice_ar/clips/` in the
repo (this batch covers all 102 sentences, including the 9 that already existed, since the voice
changed). No code or manifest changes are needed — `manifest.json` already maps every sentence to
these ids.
